In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import sys
import os
from pathlib import Path
import importlib

# Add project root to sys.path
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import and reload custom modules (to get latest version)
import src.loader
import src.analysis
importlib.reload(src.loader)
importlib.reload(src.analysis)

from src.loader import load_data, clean_data
from src.analysis import (
    calculate_driver_stats, 
    calculate_team_stats, 
    calculate_race_stats,
    calculate_combined_standings,
    calculate_combined_constructor_standings
)

# Settings
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)
print(f'Project root: {project_root}')
print('Libraries loaded successfully!')

## 1. Load All Datasets

In [ ]:
# Load Race Results
df_race = load_data('../data/Formula1_2025Season_RaceResults.csv')
df_race = clean_data(df_race)

# Load Qualifying Results
df_quali = pd.read_csv('../data/Formula1_2025Season_QualifyingResults.csv')

# Load Sprint Results
df_sprint = pd.read_csv('../data/Formula1_2025Season_SprintResults.csv')

# Load Sprint Qualifying Results
df_sprint_quali = pd.read_csv('../data/Formula1_2025Season_SprintQualifyingResults.csv')

print('All datasets loaded!')
print(f'Race Results: {df_race.shape}')
print(f'Qualifying: {df_quali.shape}')
print(f'Sprint: {df_sprint.shape}')
print(f'Sprint Qualifying: {df_sprint_quali.shape}')

## 2. Race Results Overview

In [ ]:
print('Race Results - First 10 Rows:')
df_race.head(10)

In [ ]:
print('Race Results - Info:')
df_race.info()

In [ ]:
print('Race Results - Statistics:')
df_race.describe()

## 3. Qualifying Results Overview

In [ ]:
print('Qualifying Results - First 10 Rows:')
df_quali.head(10)

In [ ]:
print('Qualifying Results - Info:')
df_quali.info()

## 4. Sprint Results Overview

In [ ]:
print('Sprint Results - First 10 Rows:')
df_sprint.head(10)

In [ ]:
print('Sprint Qualifying Results - First 10 Rows:')
df_sprint_quali.head(10)

## 5. Missing Values Analysis

In [ ]:
def check_missing_values(df, name):
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    result = pd.DataFrame({
        'Missing Values': missing,
        'Percentage': missing_pct
    })
    print(f'\n{name} - Missing Values:')
    return result[result['Missing Values'] > 0]

check_missing_values(df_race, 'Race Results')
check_missing_values(df_quali, 'Qualifying')
check_missing_values(df_sprint, 'Sprint')
check_missing_values(df_sprint_quali, 'Sprint Qualifying')

## 6. Unique Values Analysis

In [ ]:
print('Race Results - Unique Values per Column:')
for col in df_race.columns:
    print(f'{col}: {df_race[col].nunique()} unique values')

In [ ]:
print('\nList of Tracks (Race):')
print(df_race['Track'].unique())

print('\nList of Drivers:')
print(df_race['Driver'].unique())

print('\nList of Teams:')
print(df_race['Team'].unique())

## 7. Championship Standings (Race + Sprint Combined)

In [ ]:
driver_standings = calculate_combined_standings(df_race, df_sprint)
constructor_standings = calculate_combined_constructor_standings(df_race, df_sprint)

print('=== DRIVER CHAMPIONSHIP STANDINGS (Race + Sprint) ===')
driver_standings

In [ ]:
print('=== CONSTRUCTOR CHAMPIONSHIP STANDINGS (Race + Sprint) ===')
constructor_standings

In [ ]:
# Points breakdown visualization
top_10 = driver_standings.head(10).reset_index()

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Race Points',
    x=top_10['Driver'],
    y=top_10['Race_Points'],
    marker_color='#E10600'
))

fig.add_trace(go.Bar(
    name='Sprint Points',
    x=top_10['Driver'],
    y=top_10['Sprint_Points'],
    marker_color='#1E1E1E'
))

fig.update_layout(
    title='Top 10 Drivers - Points Breakdown (Race + Sprint)',
    barmode='stack',
    template='plotly_dark',
    yaxis_title='Total Points',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

## 8. Quick Visualizations

In [ ]:
# Constructor Championship Visualization
fig = go.Figure()

fig.add_trace(go.Bar(
    name='Race Points',
    x=constructor_standings.index,
    y=constructor_standings['Race_Points'],
    marker_color='#E10600'
))

fig.add_trace(go.Bar(
    name='Sprint Points',
    x=constructor_standings.index,
    y=constructor_standings['Sprint_Points'],
    marker_color='#1E1E1E'
))

fig.update_layout(
    title='Constructor Championship - Points Breakdown (Race + Sprint)',
    barmode='stack',
    template='plotly_dark',
    yaxis_title='Total Points',
    xaxis_tickangle=-45,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

In [ ]:
# Points Distribution
fig = px.histogram(df_race, x='Points', nbins=15, title='Points Distribution per Race',
                   color_discrete_sequence=['crimson'])
fig.update_layout(template='plotly_dark')
fig.show()

In [ ]:
# Position vs Grid Correlation
fig = px.scatter(df_race, x='Starting Grid', y='Position', 
                 color='Team', hover_data=['Driver', 'Track'],
                 title='Starting Grid vs Finishing Position')
fig.update_layout(template='plotly_dark')
fig.show()

---
## Summary

Notebook ini memberikan overview dasar dari semua dataset yang tersedia:
- **Race Results**: Data hasil race dengan posisi, poin, dan status
- **Qualifying**: Data qualifying dengan waktu Q1, Q2, Q3
- **Sprint**: Data hasil sprint race
- **Sprint Qualifying**: Data sprint shootout

Untuk analisis lebih detail, lihat notebook berikutnya.